<a href="https://colab.research.google.com/github/ShabbirHussainCodes/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a Ranking / Scoring task, not plain classification. The real deliverable is an ordered list — pages sorted by how much they under-perform their expected click-through rate for their search position — that an editor works through top-down, not a single yes/no label per page. Precision at a cutoff (Precision@K) is what defines "good" here, which is the classic sign of a ranking/scoring problem.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is a proxy I define, not a genuinely future-observed outcome: for each visible page, ctr_gap = that page's own CTR − the average CTR of other pages in the same position_tier. A strongly negative gap means a page is under-capturing clicks relative to its peers at the same position. It's built entirely from already-observed signals (ctr, position_tier) — never from FlyRank's own product decision flags like health_score — so it doesn't just re-learn an existing rule. It's still a proxy, not a true future outcome: a stronger version later would check whether a page's CTR actually changed after some time window, which this snapshot alone can't test causally.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50: of the top 50 pages the score ranks first, how many are "real" CTR-gap cases — meaning enough volume to trust (impressions_90d >= 500) and a meaningfully negative gap, not just noise. This matches how an editor actually uses the list: they only have time to review a limited number of pages, so what matters is whether the top of the list is trustworthy, not accuracy across all ~29,000 visible pages.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ShabbirHussainCodes/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Unit of analysis: one row = one pseudonymized content page with enough
# real visibility to trust its CTR (avg_position > 0 = has data, impressions_90d >= 500)
visible = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 500)].copy()

# The target/proxy: ctr_gap = this page's CTR minus its position tier's average CTR
visible["tier_avg_ctr"] = visible.groupby("position_tier")["ctr"].transform("mean")
visible["ctr_gap"] = visible["ctr"] - visible["tier_avg_ctr"]

print(f"Unit of analysis: {len(visible):,} visible pages (one row = one page)")

# Sketch of the target column: worst CTR-gap pages first (biggest opportunity)
cols = ["content_id", "position_tier", "impressions_90d", "ctr", "tier_avg_ctr", "ctr_gap"]
visible.sort_values("ctr_gap").loc[:, cols].head(10)

Unit of analysis: 16,726 visible pages (one row = one page)


,content_id,position_tier,impressions_90d,ctr,tier_avg_ctr,ctr_gap
8855,content_eb72d6efc172,top_3,630,0.0,0.346572,-0.346572
21301,content_ee6c6b09c17d,top_3,916,0.0,0.346572,-0.346572
11960,content_65c03925e49f,top_3,1041,0.0,0.346572,-0.346572
2288,content_c3ccadb77bc4,top_3,741,0.0,0.346572,-0.346572
26771,content_56c54a03ece2,top_3,1638,0.0,0.346572,-0.346572
26801,content_b03ea7190aea,top_3,1508,0.0,0.346572,-0.346572
22321,content_da01208f5c17,top_3,799,0.0,0.346572,-0.346572
29244,content_cca1e559cffd,top_3,1339,0.0,0.346572,-0.346572
24730,content_d0fd2e1ec8f6,top_3,519,0.0,0.346572,-0.346572
18123,content_9e6c26757e7b,top_3,1135,0.0,0.346572,-0.346572


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single hardcoded rule like "flag any page with CTR below 0.5" can't fairly compare pages across positions — position alone shifts what a "normal" CTR looks like (a top_3 page and a page_1 page have very different natural baselines), so one global number either over-flags well-behaved lower-tier pages or misses real underperformers elsewhere. Even after adjusting for position tier the way I did above, the pattern is still messier than one number can capture: content_type, main_intent, freshness, and word count all plausibly shift what "expected CTR" should look like for a page, and those factors interact rather than add up cleanly. A model — or at minimum a careful multi-factor scoring approach — can weigh several signals together and produce a more calibrated "expected CTR" per page than any single if-statement, and that gap only grows once this moves to the ~79-million-row warehouse, where the patterns are far messier than 30,000 starter rows can show.



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.